# LangGraph checkpointing on Agent Runtime: with and without Agent Gateway

**Question this notebook answers:** *Why does a LangGraph agent with a Cloud SQL (Postgres) checkpointer deploy fine on Agent Runtime, but fail once an **egress Agent Gateway** is attached, and what can you do about it?*

**Short answer (found by running this lab):** the Cloud SQL connector opens a direct mutual-TLS connection to the database, and an egress gateway in the path breaks it. Putting the database on a private IP and using VPC egress does **not** fix it. A store reached over a Google API (Firestore) does work. Ingress-only governance works too, if you set the right token-binding variable.

It is a lab: run it top to bottom. **Sections 2 and 3 contain tests that are expected to fail.** The notebook captures each failure, reads the engine logs for you and prints a plain-language diagnosis. A results table at the end summarises every experiment.

| # | Section | What it tests | Expected |
|---|---------|---------------|----------|
| 1 | Baseline | Agent Runtime + Cloud SQL checkpointer, **no gateway** | Works |
| 2 | Egress gateway + public-IP Cloud SQL | Agent Identity + gateways, with and without the token-binding opt-out | 2A, 2B: **fail** (TLS handshake). 2C (ingress only + opt-out): works |
| 3 | Egress gateway + private-IP Cloud SQL | VPC + PSC network attachment, hoping the DB connection bypasses the gateway | **Fails** (negative result, kept so you can see why) |
| 4 | Alternatives | Firestore checkpointer (HTTPS/gRPC to a Google API) | **Works** |
| 5 | Cleanup | Delete what the notebook created | n/a |
| 6 | Conclusion | Results table + recommendations | n/a |

> **Nothing is hard-coded.** The only secret is the database password, read from a Colab secret (see below).
> **Cost and time:** every deployment takes several minutes; each Cloud SQL instance bills while it exists (the cleanup section deletes them). Use a throwaway project.


## Before you start: Colab secrets and permissions

**Colab secrets** (key icon in the left sidebar). Create exactly one and turn on *Notebook access*:

| Secret name | Value | Why |
|-------------|-------|-----|
| `DB_PASSWORD` | any strong password | Password for the Postgres user the notebook creates. It is stored in Secret Manager, and the deployed agent reads it from there. It is never written into the notebook or pickled with the agent. |

Everything else (project ID, region, resource names) is plain configuration in the next cells. **You do not need to keep anything else from your original notebooks**: remove any hard-coded passwords from them.

**Permissions needed by the person running it:** Owner (or equivalent) on a project that is inside a Google Cloud **organization** (Agent Identity trust domains are organization-scoped), and the ability to create Agent Gateways (`roles/networkservices.admin`).

**Docs behind the experiments** (read these if a step surprises you):
- [Route Agent Runtime traffic through Agent Gateway](https://docs.cloud.google.com/gemini-enterprise-agent-platform/scale/runtime/agent-gateway-runtime-deploy)
- [Agent Gateway overview](https://docs.cloud.google.com/gemini-enterprise-agent-platform/govern/gateways/agent-gateway-overview)
- [Set up VPC connectivity for Agent Gateway](https://docs.cloud.google.com/gemini-enterprise-agent-platform/govern/gateways/set-up-vpc-connectivity)
- [Troubleshoot Agent Gateway connectivity](https://docs.cloud.google.com/gemini-enterprise-agent-platform/troubleshooting/troubleshoot-agent-gateway)
- [Authenticate agents with their own identity](https://docs.cloud.google.com/iam/docs/auth-agent-own-identity)

## 0. Setup

In [ ]:
%pip install -q "google-cloud-aiplatform[agent_engines,adk,langchain,langgraph,ag2,llama_index]>=1.112.0" \
    langchain-google-cloud-sql-pg "cloud-sql-python-connector[asyncpg]" \
    google-cloud-secret-manager google-cloud-firestore

In [ ]:
import os, sys, re, json, time, subprocess, asyncio, base64
from typing import Any, Iterator, Optional, Sequence

# --- authentication (Colab) ---------------------------------------------------------
from google.colab import auth, userdata
auth.authenticate_user()

# --- configuration: edit these ------------------------------------------------------
# fmt: off
PROJECT_ID   = "automation-ai-507911"   # @param {type: "string"}
REGION       = "us-central1"         # @param {type: "string"}   # Agent Runtime + Cloud SQL + gateways
MODEL        = "gemini-3.1-flash-lite"  # @param {type: "string"}
MODEL_REGION = "global"              # @param {type: "string"}
# fmt: on

if PROJECT_ID.startswith("["):
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT") or subprocess.run(
        "gcloud config get-value project", shell=True, capture_output=True, text=True).stdout.strip()
assert PROJECT_ID and not PROJECT_ID.startswith("["), "Set PROJECT_ID above."
LOCATION = REGION
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID

# --- the one secret ---------------------------------------------------------------
try:
    DB_PASSWORD = userdata.get("DB_PASSWORD")
except Exception as e:
    raise SystemExit("Create a Colab secret named DB_PASSWORD and enable notebook access. (" + str(e) + ")")
os.environ["DB_PASSWORD"] = DB_PASSWORD   # lets shell commands use "$DB_PASSWORD" without printing it

# --- resource names used throughout --------------------------------------------------
BUCKET_NAME      = f"{PROJECT_ID}-agent-staging"
SECRET_NAME      = "agent-lab-db-password"
SQL_USER         = "agentlab"
SQL_DB           = "agent-test"
SQL_PUBLIC       = "agent-lab-pg-public"      # section 1-2: public IP
SQL_PRIVATE      = "agent-lab-pg-private"     # section 3: private IP only
EGRESS_GW        = "agent-lab-egress-gw"      # section 2: plain egress gateway
INGRESS_GW       = "agent-lab-ingress-gw"
EGRESS_GW_VPC    = "agent-lab-egress-gw-vpc"  # section 3: egress gateway with a VPC (PSC) path
IAP_EXT          = "agent-lab-iap-authz-ext"
VPC_NAME         = "agent-lab-vpc"
SUBNET_NAME      = "agent-lab-psc-subnet"
SUBNET_RANGE     = "10.10.0.0/24"
PSA_RANGE_NAME   = "agent-lab-psa-range"
ATTACHMENT_NAME  = "agent-lab-gw-attachment"
TEMPLATE_NAME    = "agent-lab-egress-template"
FIRESTORE_COLL   = "langgraph_checkpoints"

def _sh(cmd, check=True, quiet=False):
    """Run a shell command. Returns (returncode, combined output). Never echoes the command itself."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, executable="/bin/bash")
    out = (r.stdout + r.stderr).strip()
    if out and not quiet:
        print(out)
    if check and r.returncode != 0:
        raise RuntimeError(f"command failed (exit {r.returncode}): {cmd[:120]}")
    return r.returncode, out

def _exists(cmd):
    return _sh(cmd, check=False, quiet=True)[0] == 0

_sh(f"gcloud config set project {PROJECT_ID}", quiet=True)
PROJECT_NUMBER = _sh(f"gcloud projects describe {PROJECT_ID} --format='value(projectNumber)'", quiet=True)[1]
_parent_type = _sh(f"gcloud projects describe {PROJECT_ID} --format='value(parent.type)'", quiet=True)[1].strip()
ORG_ID       = _sh(f"gcloud projects describe {PROJECT_ID} --format='value(parent.id)'", quiet=True)[1].strip()
assert _parent_type == "organization" and ORG_ID.isdigit(), (
    f"Project parent is '{_parent_type}' / '{ORG_ID}'. Agent Identity needs the project inside an organization.")

TRUST_DOMAIN    = f"agents.global.org-{ORG_ID}.system.id.goog"
AGENT_PRINCIPAL = f"principalSet://{TRUST_DOMAIN}/attribute.platformContainer/aiplatform/projects/{PROJECT_NUMBER}"
RE_SERVICE_AGENT = f"serviceAccount:service-{PROJECT_NUMBER}@gcp-sa-aiplatform-re.iam.gserviceaccount.com"
GW_SERVICE_AGENT = f"serviceAccount:service-{PROJECT_NUMBER}@gcp-sa-agentgateway.iam.gserviceaccount.com"

print("project      :", PROJECT_ID, f"({PROJECT_NUMBER})")
print("organization :", ORG_ID)
print("agent identity principal:\n ", AGENT_PRINCIPAL)

In [ ]:
import vertexai
from vertexai import agent_engines, types

vertexai.init(project=PROJECT_ID, location=MODEL_REGION)
client = vertexai.Client(project=PROJECT_ID, location=LOCATION)   # (newer SDKs may prefer agentplatform.Client)

# ---------------------------------------------------------------------------------
# Helpers shared by every experiment
# ---------------------------------------------------------------------------------
RESULTS = {}     # experiment label -> (status, diagnosis)
ENGINES = []     # reasoning-engine IDs created by this notebook (so cleanup only touches our own)

# (regex on engine logs, explanation). First match wins.
SIGNATURES = [
    (r"401.*(invalid authentication credentials|sqladmin|secretmanager)|invalid authentication credentials",
     "401 from a Google API: the Agent Identity token is bound to the runtime (DPoP / mTLS Context-Aware Access) "
     "and the client library sent it as a plain bearer token. Fix: set env var GOOGLE_API_PREVENT_AGENT_TOKEN_SHARING_FOR_GCP_SERVICES=false "
     "(documented opt-out; weakens token-theft protection)."),
    (r"SSLV3_ALERT_HANDSHAKE_FAILURE|ssl\.SSLError|CERTIFICATE_VERIFY_FAILED",
     "TLS handshake failure on the DATABASE connection. The Cloud SQL connector needs end-to-end mutual TLS to the "
     "instance; the egress gateway terminates and re-signs TLS, so the client certificate never reaches Cloud SQL. "
     "Fix: take the DB connection off the gateway path (private IP + VPC egress) or use an HTTP-based store."),
    (r"Egress request is not authorized|\b403\b",
     "403: egress is default-deny. Register the exact hostname and grant the agent identity iap.resources.egressViaIAP, "
     "or keep the IAP extension in DRY_RUN."),
    (r"\b498\b",
     "498: gateway bound before the destination was allow-listed. Allow-list first or use DRY_RUN."),
    (r"TimeoutError|timed out|Connect call failed|Network is unreachable|No route to host",
     "Network timeout: no route from the runtime to the destination (VPC/PSC path, DNS peering, firewall, or the "
     "destination is not reachable from the attachment subnet)."),
    (r"password authentication failed",
     "Wrong DB user/password. Check the Secret Manager secret and the Cloud SQL user."),
    (r"relation .* does not exist|does not exist",
     "Checkpoint tables are missing. engine.init_checkpoint_table() must run once."),
    (r"PermissionDenied|permission denied|does not have .* permission",
     "IAM permission missing for the runtime identity; see the exact permission in the log line."),
]

def engine_id_from(text):
    m = re.search(r"reasoningEngines/(\d+)", str(text))
    return m.group(1) if m else None

def fetch_logs(engine_id, limit=300, wait_s=45):
    """Pull recent logs for a reasoning engine. Logs lag by ~30-60s, so wait first."""
    time.sleep(wait_s)
    flt = ('resource.type="aiplatform.googleapis.com/ReasoningEngine" '
           f'AND resource.labels.reasoning_engine_id="{engine_id}"')
    _, out = _sh(f"gcloud logging read '{flt}' --project={PROJECT_ID} --limit={limit} "
                 f"--freshness=1d --format='value(textPayload)'", check=False, quiet=True)
    return out

def diagnose(log_text):
    for pattern, explanation in SIGNATURES:
        if re.search(pattern, log_text, flags=re.S):
            return explanation
    return "No known signature matched. Read the log excerpt above (look at the last traceback)."

def last_traceback(log_text, n=25):
    lines = [l for l in log_text.splitlines() if l.strip()]
    tb = [i for i, l in enumerate(lines) if "Traceback" in l]
    start = tb[-1] if tb else max(0, len(lines) - n)
    return "\n".join(lines[start:start + n])

def deploy(label, agent, config):
    """Create an Agent Runtime engine. On failure: find the engine, read its logs, diagnose, record."""
    print(f"\n=== DEPLOY: {label} ===")
    try:
        remote = client.agent_engines.create(agent=agent, config=config)
        ENGINES.append(engine_id_from(remote.api_resource.name))
        RESULTS[label] = ("DEPLOYED", "")
        print("OK ->", remote.api_resource.name)
        return remote
    except Exception as e:
        eid = engine_id_from(e)
        if eid:
            ENGINES.append(eid)
        print("DEPLOY FAILED:", str(e)[:300])
        diag = "Could not find the engine id in the error."
        if eid:
            print(f"Reading logs for engine {eid} (waiting ~45s for log ingestion)...")
            logs = fetch_logs(eid)
            print("--- last traceback in logs ---")
            print(last_traceback(logs) or "(no logs yet; re-run fetch_logs)")
            diag = diagnose(logs)
        print("\nDIAGNOSIS:", diag)
        RESULTS[label] = ("DEPLOY FAILED", diag)
        return None

def ask(remote, text="scrum methodology", thread_id="lab-thread-1", label=None):
    """Query a deployed agent twice on one thread, then try to read the persisted state back."""
    cfg = {"configurable": {"thread_id": thread_id}}
    try:
        out = remote.query(input={"role": "user", "content": text}, config=cfg)
        print("query #1 ->", str(out)[:300].replace("\n", " "), "...")
        out = remote.query(input={"role": "user", "content": "now summarise that in one line"}, config=cfg)
        print("query #2 ->", str(out)[:300].replace("\n", " "), "...")
    except Exception as e:
        print("QUERY FAILED:", str(e)[:500])
        if label:
            RESULTS[label] = ("DEPLOYED, QUERY FAILED", str(e)[:200])
        return
    try:
        hist = remote.get_state_history(config=cfg)
        n = len(list(hist)) if hist is not None else 0
        print(f"checkpoints visible for thread '{thread_id}': {n}")
    except Exception as e:
        print("(could not read state history through the agent API:", str(e)[:200], ")")
    if label:
        RESULTS[label] = ("WORKS", "deployed, answered two turns on one thread")

def cleanup_engines():
    """Delete only the reasoning engines THIS notebook created (tracked in ENGINES)."""
    for eid in list(ENGINES):
        name = f"projects/{PROJECT_NUMBER}/locations/{LOCATION}/reasoningEngines/{eid}"
        try:
            client.agent_engines.delete(name=name, force=True)
            print("deleted engine", eid)
        except Exception as e:
            print("could not delete", eid, "->", str(e)[:150])
        ENGINES.remove(eid)

def gw_path(name):
    return f"projects/{PROJECT_ID}/locations/{REGION}/agentGateways/{name}"

REQUIREMENTS = [
    "google-cloud-aiplatform[agent_engines,adk,langchain,langgraph,ag2,llama_index]>=1.112.0",
    "langchain-google-cloud-sql-pg",
    "google-cloud-secret-manager",
    "google-cloud-firestore",
    # tip: pin exact versions once you have a working combination
]

def deploy_config(display_name, egress_gw=None, ingress_gw=None, agent_identity=True, env=None):
    cfg = {
        "requirements": REQUIREMENTS,
        "staging_bucket": f"gs://{BUCKET_NAME}",
        "display_name": display_name,
        "agent_framework": "langgraph",
    }
    if agent_identity:
        cfg["identity_type"] = types.IdentityType.AGENT_IDENTITY
    gwc = {}
    if ingress_gw:
        gwc["client_to_agent_config"] = {"agent_gateway": gw_path(ingress_gw)}
    if egress_gw:
        gwc["agent_to_anywhere_config"] = {"agent_gateway": gw_path(egress_gw)}
    if gwc:
        cfg["agent_gateway_config"] = gwc
    if env:
        cfg["env_vars"] = env
    return cfg

# The documented opt-out for token binding on Agent Identity credentials.
# https://docs.cloud.google.com/iam/docs/auth-agent-own-identity
TOKEN_BINDING_OFF = {"GOOGLE_API_PREVENT_AGENT_TOKEN_SHARING_FOR_GCP_SERVICES": "false"}
print("helpers ready")

---
## Section 1: Baseline: Agent Runtime + Cloud SQL checkpointer, no gateway

This reproduces the setup that works: a LangGraph agent whose checkpointer is `PostgresSaver` on Cloud SQL, deployed to Agent Runtime with the default service identity and **no gateway**.

Two changes from a typical tutorial, both so the notebook is safe to share:
- the DB password lives in **Secret Manager** and the deployed agent fetches it at start-up (nothing secret is pickled into the staging bucket);
- the checkpointer builder calls `init_checkpoint_table()` (create-if-missing) so the tables always exist.

In [ ]:
# Enable APIs, create the staging bucket, grant the runtime what it needs
_sh("gcloud services enable aiplatform.googleapis.com sqladmin.googleapis.com secretmanager.googleapis.com "
    "compute.googleapis.com servicenetworking.googleapis.com dns.googleapis.com firestore.googleapis.com "
    "networkservices.googleapis.com networksecurity.googleapis.com iap.googleapis.com "
    "agentregistry.googleapis.com certificatemanager.googleapis.com cloudresourcemanager.googleapis.com "
    "logging.googleapis.com")

if not _exists(f"gcloud storage buckets describe gs://{BUCKET_NAME}"):
    _sh(f"gcloud storage buckets create gs://{BUCKET_NAME} --location={LOCATION} --project={PROJECT_ID}")

def grant(member, role):
    rc, out = _sh(f"gcloud projects add-iam-policy-binding {PROJECT_ID} --member='{member}' "
                  f"--role='{role}' --condition=None", check=False, quiet=True)
    print(("ok   " if rc == 0 else "SKIP ") + role + ("" if rc == 0 else "  (" + out.splitlines()[-1][:90] + ")"))

# Default service identity of Agent Runtime (used in section 1)
for role in ["roles/cloudsql.client", "roles/secretmanager.secretAccessor"]:
    grant(RE_SERVICE_AGENT, role)

# Agent Identity principal (used in sections 2-4). Some of these role names come from the troubleshooting
# page; if one does not exist in your environment it is skipped, which is fine.
for role in ["roles/cloudsql.client", "roles/secretmanager.secretAccessor", "roles/serviceusage.serviceUsageConsumer",
             "roles/aiplatform.user", "roles/logging.logWriter", "roles/monitoring.metricWriter",
             "roles/browser", "roles/agentregistry.viewer", "roles/aiplatform.agentDefaultAccess",
             "roles/datastore.user"]:
    grant(AGENT_PRINCIPAL, role)

In [ ]:
# Public-IP Cloud SQL instance + database + user + secret  (takes ~5-10 minutes)
if not _exists(f"gcloud sql instances describe {SQL_PUBLIC}"):
    _sh(f"gcloud sql instances create {SQL_PUBLIC} --database-version=POSTGRES_15 --edition=ENTERPRISE "
        f"--tier=db-f1-micro --region={REGION}")
if not _exists(f"gcloud sql databases describe {SQL_DB} --instance={SQL_PUBLIC}"):
    _sh(f"gcloud sql databases create {SQL_DB} --instance={SQL_PUBLIC}")
if _sh(f"gcloud sql users list --instance={SQL_PUBLIC} --format='value(name)'", quiet=True)[1].split().count(SQL_USER):
    _sh(f"gcloud sql users set-password {SQL_USER} --instance={SQL_PUBLIC} --password=\"$DB_PASSWORD\"", quiet=True)
else:
    _sh(f"gcloud sql users create {SQL_USER} --instance={SQL_PUBLIC} --password=\"$DB_PASSWORD\"", quiet=True)

# Secret Manager (the password is piped via stdin, never on a command line or in the notebook)
if not _exists(f"gcloud secrets describe {SECRET_NAME}"):
    _sh(f"gcloud secrets create {SECRET_NAME} --replication-policy=automatic", quiet=True)
subprocess.run(["gcloud", "secrets", "versions", "add", SECRET_NAME, "--data-file=-", f"--project={PROJECT_ID}"],
               input=DB_PASSWORD, text=True, capture_output=True, check=True)
print("public instance + secret ready")

In [ ]:
# ---- the agent: same graph as the original notebook ----------------------------------------------
def langgraph_builder(*, model, checkpointer=None,**kwargs):
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_core.output_parsers import StrOutputParser
    from langgraph.graph import END, MessageGraph    # deprecated upstream (use StateGraph), kept to match the original

    output_parser = StrOutputParser()
    planner = ChatPromptTemplate.from_template("Generate an argument about: {input}") | model | output_parser
    pros    = ChatPromptTemplate.from_template("List the positive aspects of {input}") | model | output_parser
    cons    = ChatPromptTemplate.from_template("List the negative aspects of {input}") | model | output_parser
    summary = ChatPromptTemplate.from_template(
        "Input:{input}\nGenerate a final response given the critique") | model | output_parser

    builder = MessageGraph()
    builder.add_node("planner", planner)
    builder.add_node("pros", pros)
    builder.add_node("cons", cons)
    builder.add_node("summary", summary)
    builder.add_edge("planner", "pros")
    builder.add_edge("planner", "cons")
    builder.add_edge("pros", "summary")
    builder.add_edge("cons", "summary")
    builder.add_edge("summary", END)
    builder.set_entry_point("planner")
    return builder.compile(checkpointer=checkpointer)

# ---- Cloud SQL checkpointer builder (runs INSIDE the deployed runtime, so it must be self-contained) ----
def pg_checkpointer_builder(**kwargs):
    from google.cloud import secretmanager
    from langchain_google_cloud_sql_pg import PostgresEngine, PostgresSaver

    # REST transport = plain HTTPS, friendlier to an HTTP gateway than gRPC
    sm = secretmanager.SecretManagerServiceClient(transport="rest")
    password = sm.access_secret_version(
        name=f"projects/{kwargs['project_id']}/secrets/{kwargs['secret_id']}/versions/latest"
    ).payload.data.decode()

    engine = PostgresEngine.from_instance(
        project_id=kwargs["project_id"], region=kwargs["region"],
        instance=kwargs["instance"], database=kwargs["database"],
        user=kwargs["user"], password=password,
        ip_type=kwargs.get("ip_type", "PUBLIC"),
    )
    try:
        engine.init_checkpoint_table()      # create-if-missing
    except Exception as e:
        print("init_checkpoint_table skipped:", repr(e)[:200])
    return PostgresSaver.create_sync(engine)

def make_agent(checkpointer_builder, checkpointer_kwargs):
    return agent_engines.LanggraphAgent(
        model=MODEL,
        runnable_builder=langgraph_builder,
        checkpointer_kwargs=checkpointer_kwargs,
        checkpointer_builder=checkpointer_builder,
    )

PG_PUBLIC_KWARGS = dict(project_id=PROJECT_ID, region=REGION, instance=SQL_PUBLIC, database=SQL_DB,
                        user=SQL_USER, secret_id=SECRET_NAME, ip_type="PUBLIC")
agent_pg_public = make_agent(pg_checkpointer_builder, PG_PUBLIC_KWARGS)
print("agent object built")

In [ ]:
# Local smoke test (runs in Colab using YOUR credentials; Colab can reach the public-IP instance)
cfg = {"configurable": {"thread_id": "local-1"}}
out = agent_pg_public.query(input={"role": "user", "content": "scrum methodology"}, config=cfg)
print(str(out)[:300].replace("\n", " "), "...")
RESULTS["1. local run, Cloud SQL checkpointer"] = ("WORKS", "Colab -> public IP")

In [ ]:
from IPython.display import display, Markdown

# Render a simple Markdown string
display(Markdown(out[-1]['kwargs']['content']))

In [ ]:
# Deploy to Agent Runtime with NO gateway and the default service identity
remote_baseline = deploy(
    "1. deployed, Cloud SQL checkpointer, no gateway",
    agent_pg_public,
    deploy_config("lab-baseline", agent_identity=False),
)
if remote_baseline:
    ask(remote_baseline, thread_id="baseline-1", label="1. deployed, Cloud SQL checkpointer, no gateway")

---
## Section 2: Egress gateway + public-IP Cloud SQL (the failing case)

Now we put the agent behind **Agent Gateway** and switch it to **Agent Identity** (a gateway requires it for governance).

We run three deployments of the *same* agent against the *same* public-IP database:

| Test | Gateways | Token-binding opt-out | Expected |
|------|----------|-----------------------|----------|
| 2A | ingress + egress | no | **TLS handshake failure** on the DB connection |
| 2B | ingress + egress | yes | **TLS handshake failure**, identical to 2A |
| 2C | ingress only | yes | Works (the DB connection is not on the gateway path) |

The opt-out variable is `GOOGLE_API_PREVENT_AGENT_TOKEN_SHARING_FOR_GCP_SERVICES=false`. It only matters when there is **no** egress gateway: a run with ingress only and the documented-looking `GOOGLE_API_ENABLE_RUNTIME_BOUND_TOKEN=false` still gets a `401` from `sqladmin`; this was tested.

First, create the gateways. The IAP authorization extension is kept in `DRY_RUN`, so egress is observed and not blocked. That keeps the focus on the checkpointer.


In [ ]:
# Egress + ingress gateways, IAP extension (DRY_RUN) and authz policy
open("egress-gw.yaml", "w").write(f"""name: {gw_path(EGRESS_GW)}
protocols:
  - MCP
googleManaged:
  governedAccessPath: AGENT_TO_ANYWHERE
registries:
  - //agentregistry.googleapis.com/projects/{PROJECT_ID}/locations/{REGION}
""")
open("ingress-gw.yaml", "w").write(f"""name: {gw_path(INGRESS_GW)}
protocols:
  - MCP
googleManaged:
  governedAccessPath: CLIENT_TO_AGENT
""")
open("iap-ext.yaml", "w").write(f"""name: {IAP_EXT}
service: iap.googleapis.com
failOpen: false
timeout: 1s
metadata:
  iapPolicyVersion: "V2"
  iamEnforcementMode: "DRY_RUN"
""")
open("iap-policy.yaml", "w").write(f"""name: {EGRESS_GW}-authz-policy
target:
  resources:
    - "{gw_path(EGRESS_GW)}"
policyProfile: REQUEST_AUTHZ
action: CUSTOM
customProvider:
  authzExtension:
    resources:
      - "projects/{PROJECT_ID}/locations/{REGION}/authzExtensions/{IAP_EXT}"
""")

_sh(f"gcloud network-services agent-gateways import {EGRESS_GW} --source=egress-gw.yaml --location={REGION} --project={PROJECT_ID}")
_sh(f"gcloud network-services agent-gateways import {INGRESS_GW} --source=ingress-gw.yaml --location={REGION} --project={PROJECT_ID}")
_sh(f"gcloud beta service-extensions authz-extensions import {IAP_EXT} --source=iap-ext.yaml --location={REGION} --project={PROJECT_ID}")
_sh(f"gcloud network-security authz-policies import {EGRESS_GW}-authz-policy --source=iap-policy.yaml --location={REGION} --project={PROJECT_ID}")
print("gateways ready")

In [ ]:
# 2A: Agent Identity + ingress + egress, token binding ON (default)   -> expect: SSL handshake failure (the egress gateway is in the path)
cleanup_engines()
remote_2a = deploy("2A. egress+ingress, Agent Identity, token binding ON", agent_pg_public,
                   deploy_config("lab-2a", egress_gw=EGRESS_GW, ingress_gw=INGRESS_GW))

In [ ]:
# 2B: same, but with the token-binding opt-out   -> expect: SSL handshake failure on the DB connection (same as 2A)
remote_2b = deploy("2B. egress+ingress, Agent Identity, token binding OFF", agent_pg_public,
                   deploy_config("lab-2b", egress_gw=EGRESS_GW, ingress_gw=INGRESS_GW, env=TOKEN_BINDING_OFF))

In [ ]:
# 2C: ingress gateway only (no egress governance) + opt-out   -> expect: works (needs the PREVENT_AGENT_TOKEN_SHARING variable; the RUNTIME_BOUND_TOKEN one gives 401)
# NOTE: Agent Runtime allows one egress and one ingress gateway per project+region; engines from 2A/2B that
# failed to deploy do not serve traffic, but delete them if the platform complains about bindings.
remote_2c = deploy("2C. ingress only, Agent Identity, token binding OFF", agent_pg_public,
                   deploy_config("lab-2c", ingress_gw=INGRESS_GW, env=TOKEN_BINDING_OFF))
if remote_2c:
    ask(remote_2c, thread_id="2c-1", label="2C. ingress only, Agent Identity, token binding OFF")

### What section 2 shows

1. **2A and 2B fail the same way, at `asyncpg.connect`, with `SSLV3_ALERT_HANDSHAKE_FAILURE`.** The connector first calls `sqladmin` for connection info and a client certificate (that call succeeds when an egress gateway is in the path), then opens a direct **mutual-TLS** connection to the instance on port 3307. The gateway performs *TLS decryption and inspection* on outbound traffic, and a TLS-terminating proxy in the middle breaks that session. The token-binding variable makes no difference here.
2. **2C works** because ingress governance does not touch outbound traffic. Same agent, same DB, no egress gateway in the path.
3. **2C also shows the token-binding problem.** Without an egress gateway, the Agent Identity token is bound to the runtime (DPoP / mTLS Context-Aware Access) and the Cloud SQL connector's HTTP client is not aware of it, so `sqladmin.googleapis.com/.../connectSettings` returns `401`. In our tests:
   - `GOOGLE_API_PREVENT_AGENT_TOKEN_SHARING_FOR_GCP_SERVICES=false` → **works**;
   - `GOOGLE_API_ENABLE_RUNTIME_BOUND_TOKEN=false` (the one that looks documented) → **still 401**.
   Both weaken protection against stolen tokens, so treat this as a learning aid, not a production default.

So the failing combination is **egress gateway + a direct database wire connection**. Not Postgres, not LangGraph, not the checkpointer's code.


---
## Section 3: Egress gateway + private-IP Cloud SQL (negative result: does NOT bypass the gateway)

**Hypothesis (from the docs):** an egress gateway can be given a **PSC network attachment** with `vpcEgress: PRIVATE_RANGES_ONLY`. Traffic to private (RFC 1918) addresses would be sent *into your VPC* through the attachment, while public destinations (such as `sqladmin.googleapis.com`) still go through the gateway. With a private-IP-only database and the connector using `ip_type="PRIVATE"`, the DB connection should avoid the gateway's TLS inspection.

**What we found:** it does not. The connector does use the private IP, but the deployment fails with the same `SSLV3_ALERT_HANDSHAKE_FAILURE`. In a diagnostic run, TCP connections to **unused** ports on the private IP were also accepted (from a link-local source address `169.254.x.x`, not your subnet). A real network refuses those, so the traffic appears to be terminated by a proxy local to the runtime and never reaches your VPC. This is an inference from probes, not a confirmed root cause. `ALL_TRAFFIC` mode was not tried.

The section is kept as a documented negative result: you can re-run it to check whether a newer platform version behaves differently.

Steps: VPC and subnet, private services access, a **new** private-IP Cloud SQL instance, PSC attachment, a connectivity template, a **new** egress gateway that uses it (a gateway created without a template cannot be edited afterwards), then a deployment that is expected to fail.


In [ ]:
# Free the one-egress-gateway-per-project slot: delete engines this notebook created earlier.
_sh(f"gcloud network-security authz-policies delete {EGRESS_GW}-authz-policy --location={REGION} --quiet", check=False)
_sh(f"gcloud network-services agent-gateways delete {EGRESS_GW} --location={REGION} --quiet", check=False)

cleanup_engines()   # engines bound to a gateway can block its deletion



# VPC + subnet (Private Google Access is required on the attachment subnet)
if not _exists(f"gcloud compute networks describe {VPC_NAME}"):
    _sh(f"gcloud compute networks create {VPC_NAME} --subnet-mode=custom")
if not _exists(f"gcloud compute networks subnets describe {SUBNET_NAME} --region={REGION}"):
    _sh(f"gcloud compute networks subnets create {SUBNET_NAME} --network={VPC_NAME} --region={REGION} "
        f"--range={SUBNET_RANGE} --enable-private-ip-google-access")

# Private services access: lets Cloud SQL get a private IP in this VPC
if not _exists(f"gcloud compute addresses describe {PSA_RANGE_NAME} --global"):
    _sh(f"gcloud compute addresses create {PSA_RANGE_NAME} --global --purpose=VPC_PEERING "
        f"--prefix-length=20 --network={VPC_NAME}")
rc, out = _sh(f"gcloud services vpc-peerings connect --service=servicenetworking.googleapis.com "
              f"--ranges={PSA_RANGE_NAME} --network={VPC_NAME}", check=False)
print("peering ok" if rc == 0 else "peering call returned an error (often 'already exists'); continuing")

In [ ]:
# NEW Cloud SQL instance with a PRIVATE IP only (several minutes)
if not _exists(f"gcloud sql instances describe {SQL_PRIVATE}"):
    _sh(f"gcloud sql instances create {SQL_PRIVATE} --database-version=POSTGRES_15 --edition=ENTERPRISE "
        f"--tier=db-f1-micro --region={REGION} "
        f"--network=projects/{PROJECT_ID}/global/networks/{VPC_NAME} --no-assign-ip")
if not _exists(f"gcloud sql databases describe {SQL_DB} --instance={SQL_PRIVATE}"):
    _sh(f"gcloud sql databases create {SQL_DB} --instance={SQL_PRIVATE}")
if _sh(f"gcloud sql users list --instance={SQL_PRIVATE} --format='value(name)'", quiet=True)[1].split().count(SQL_USER):
    _sh(f"gcloud sql users set-password {SQL_USER} --instance={SQL_PRIVATE} --password=\"$DB_PASSWORD\"", quiet=True)
else:
    _sh(f"gcloud sql users create {SQL_USER} --instance={SQL_PRIVATE} --password=\"$DB_PASSWORD\"", quiet=True)

print("private IP:", _sh(f"gcloud sql instances describe {SQL_PRIVATE} --format='value(ipAddresses[0].ipAddress)'", quiet=True)[1])
print("Colab cannot reach this instance (no public IP). Tables are created by the deployed agent on first start.")

In [ ]:
# PSC network attachment, connectivity template, new egress gateway, service-agent roles, authz policy
if not _exists(f"gcloud compute network-attachments describe {ATTACHMENT_NAME} --region={REGION}"):
    _sh(f"gcloud compute network-attachments create {ATTACHMENT_NAME} --region={REGION} "
        f"--subnets={SUBNET_NAME} --connection-preference=ACCEPT_AUTOMATIC")
ATTACHMENT_URI = f"projects/{PROJECT_ID}/regions/{REGION}/networkAttachments/{ATTACHMENT_NAME}"

# vpcEgress: PRIVATE_RANGES_ONLY = private IPs go into the VPC, public destinations still use the gateway.
# (ALL_TRAFFIC would send everything via the VPC and needs a default route + Cloud NAT.)
open("egress-template.yaml", "w").write(f"""name: projects/{PROJECT_NUMBER}/locations/{REGION}/agentConnectivityTemplates/{TEMPLATE_NAME}
accessPath: AGENT_TO_ANYWHERE
deploymentModel: CENTRALIZED
egressNetworkConfig:
  networkAttachment: {ATTACHMENT_URI}
  vpcEgress: PRIVATE_RANGES_ONLY
""")
_sh(f"gcloud network-services agent-connectivity-templates import {TEMPLATE_NAME} "
    f"--source=egress-template.yaml --location={REGION}")

open("egress-gw-vpc.yaml", "w").write(f"""name: {gw_path(EGRESS_GW_VPC)}
protocols:
  - MCP
googleManaged:
  governedAccessPath: AGENT_TO_ANYWHERE
agentConnectivityTemplate: projects/{PROJECT_NUMBER}/locations/{REGION}/agentConnectivityTemplates/{TEMPLATE_NAME}
registries:
  - //agentregistry.googleapis.com/projects/{PROJECT_ID}/locations/{REGION}
""")
_sh(f"gcloud network-services agent-gateways import {EGRESS_GW_VPC} --source=egress-gw-vpc.yaml --location={REGION}")

# The gateway's service agent must be able to use the attachment (it exists once a gateway exists)
for role in ["roles/compute.networkUser", "roles/dns.peer"]:
    grant(GW_SERVICE_AGENT, role)

open("iap-policy-vpc.yaml", "w").write(f"""name: {EGRESS_GW_VPC}-authz-policy
target:
  resources:
    - "{gw_path(EGRESS_GW_VPC)}"
policyProfile: REQUEST_AUTHZ
action: CUSTOM
customProvider:
  authzExtension:
    resources:
      - "projects/{PROJECT_ID}/locations/{REGION}/authzExtensions/{IAP_EXT}"
""")
_sh(f"gcloud network-security authz-policies import {EGRESS_GW_VPC}-authz-policy --source=iap-policy-vpc.yaml --location={REGION}")
print("VPC egress gateway ready")

In [ ]:
# EXPECTED TO FAIL (documented negative result). Deploy the same agent against the PRIVATE instance, behind the VPC-egress gateway + ingress gateway
PG_PRIVATE_KWARGS = dict(project_id=PROJECT_ID, region=REGION, instance=SQL_PRIVATE, database=SQL_DB,
                         user=SQL_USER, secret_id=SECRET_NAME, ip_type="PRIVATE")
agent_pg_private = make_agent(pg_checkpointer_builder, PG_PRIVATE_KWARGS)

remote_3 = deploy("3. private-IP Cloud SQL + VPC egress gateway + ingress, token binding OFF", agent_pg_private,
                  deploy_config("lab-3", egress_gw=EGRESS_GW_VPC, ingress_gw=INGRESS_GW, env=TOKEN_BINDING_OFF))
if remote_3:
    ask(remote_3, thread_id="3-1", label="3. private-IP Cloud SQL + VPC egress gateway + ingress, token binding OFF")

### Reading the result of section 3

| What you see | What it means | Next step |
|--------------|---------------|-----------|
| `SSLV3_ALERT_HANDSHAKE_FAILURE` | **Expected (what we observed).** The private-range path did not take the connection out of the gateway's TLS handling. | Go to section 4 (Google-API stores). |
| **Deploys and answers two turns** | Unexpected: the platform now lets private-range TCP bypass inspection. | The Cloud SQL checkpointer can coexist with an egress gateway; keep this design and tell us. |
| Timeout / `Connect call failed` | No route from the attachment subnet to the Cloud SQL private range (peering/route/firewall/DNS). | Check the attachment subnet and the peering routes. |
| `401` | A token-binding variable did not apply. | Confirm `env_vars` shows `GOOGLE_API_PREVENT_AGENT_TOKEN_SHARING_FOR_GCP_SERVICES=false`. |
| `403` / `498` | Egress is default-deny: `sqladmin.googleapis.com` (and `secretmanager.googleapis.com`) need to be registered and the agent identity needs `iap.resources.egressViaIAP`. | See the Troubleshoot Agent Gateway page. |

The cell above already printed the last traceback and the matching diagnosis.


## Section 4: Alternatives that do not open a database wire connection

Since the direct DB connection cannot coexist with the egress gateway, use a store reached over **HTTPS/gRPC to a Google API**, which the gateway can carry.

### 4.1 Firestore as a LangGraph checkpointer (works behind the egress gateway)

There is no first-party Google Firestore checkpointer for LangGraph in Google's Agent Runtime docs, so this notebook includes a **small self-contained saver**. It implements LangGraph's `BaseCheckpointSaver` contract (`get_tuple`, `list`, `put`, `put_writes`, `delete_thread`, plus async wrappers). It was tested locally against a fake Firestore and deployed behind an egress gateway plus ingress gateway, where it stored and returned checkpoints.

Limits to know: a Firestore document is capped at about 1 MiB, so a single checkpoint (for example a very long message history) can exceed it. Newer LangGraph base-class methods (`copy_thread`, `prune`) are not implemented. It is a reference implementation, not load-tested and not a Google product.


In [ ]:
# Firestore database (Native mode) + a local smoke test of the saver
if not _exists(f"gcloud firestore databases describe --database='(default)' --project={PROJECT_ID}"):
    _sh(f"gcloud firestore databases create --database='(default)' --location={REGION} --type=firestore-native "
        f"--project={PROJECT_ID}", check=False)
print("firestore ready (or already existed)")

In [ ]:
from langchain_core.runnables import RunnableConfig
from langgraph.checkpoint.base import (
    BaseCheckpointSaver, Checkpoint, CheckpointMetadata, CheckpointTuple, get_checkpoint_id,
)
try:
    from langgraph.checkpoint.base import get_checkpoint_metadata, WRITES_IDX_MAP
except ImportError:                                            # very old/new versions: degrade gracefully
    get_checkpoint_metadata = lambda config, metadata: metadata
    WRITES_IDX_MAP = {}


class FirestoreSaver(BaseCheckpointSaver):
    """LangGraph checkpointer backed by Cloud Firestore (reference implementation).

    Layout:  <collection>/<thread>/ns/<checkpoint_ns>/checkpoints/<checkpoint_id>
             <collection>/<thread>/ns/<checkpoint_ns>/writes/<checkpoint_id>__<task_id>__<idx>
    """

    def __init__(self, project_id: str, database: str = "(default)", collection: str = "langgraph_checkpoints"):
        super().__init__()
        from google.cloud import firestore
        self._fs = firestore
        self.db = firestore.Client(project=project_id, database=database)
        self.collection = collection

    # ---- helpers ----
    @staticmethod
    def _k(s: str) -> str:
        """Firestore document ids cannot contain '/', so encode ids."""
        return base64.urlsafe_b64encode(s.encode()).decode() or "_"

    def _ns_ref(self, thread_id: str, ns: str):
        return (self.db.collection(self.collection).document(self._k(thread_id))
                .collection("ns").document(self._k(ns)))

    def _tuple(self, thread_id: str, ns: str, doc: dict) -> CheckpointTuple:
        checkpoint = self.serde.loads_typed((doc["type"], doc["checkpoint"]))
        metadata = self.serde.loads_typed((doc["metadata_type"], doc["metadata"]))
        base = self._ns_ref(thread_id, ns)
        wdocs = [w.to_dict() for w in base.collection("writes").where("checkpoint_id", "==", doc["checkpoint_id"]).stream()]
        wdocs.sort(key=lambda w: (w["task_id"], w["idx"]))
        pending = [(w["task_id"], w["channel"], self.serde.loads_typed((w["type"], w["value"]))) for w in wdocs]
        parent = None
        if doc.get("parent_checkpoint_id"):
            parent = {"configurable": {"thread_id": thread_id, "checkpoint_ns": ns,
                                       "checkpoint_id": doc["parent_checkpoint_id"]}}
        return CheckpointTuple(
            {"configurable": {"thread_id": thread_id, "checkpoint_ns": ns, "checkpoint_id": doc["checkpoint_id"]}},
            checkpoint, metadata, parent, pending,
        )

    # ---- sync API ----
    def get_tuple(self, config: RunnableConfig) -> Optional[CheckpointTuple]:
        c = config["configurable"]
        thread_id, ns = c["thread_id"], c.get("checkpoint_ns", "")
        cid = get_checkpoint_id(config)
        col = self._ns_ref(thread_id, ns).collection("checkpoints")
        if cid:
            snap = col.document(cid).get()
            doc = snap.to_dict() if snap.exists else None
        else:
            docs = list(col.order_by("checkpoint_id", direction=self._fs.Query.DESCENDING).limit(1).stream())
            doc = docs[0].to_dict() if docs else None
        return self._tuple(thread_id, ns, doc) if doc else None

    def list(self, config: Optional[RunnableConfig], *, filter: Optional[dict] = None,
             before: Optional[RunnableConfig] = None, limit: Optional[int] = None) -> Iterator[CheckpointTuple]:
        if config is None:                       # listing across all threads is not supported here
            return
        c = config["configurable"]
        thread_id, ns = c["thread_id"], c.get("checkpoint_ns", "")
        q = self._ns_ref(thread_id, ns).collection("checkpoints").order_by(
            "checkpoint_id", direction=self._fs.Query.DESCENDING)
        if before and get_checkpoint_id(before):
            q = q.where("checkpoint_id", "<", get_checkpoint_id(before))
        n = 0
        for snap in q.stream():
            t = self._tuple(thread_id, ns, snap.to_dict())
            if filter and not all(t.metadata.get(k) == v for k, v in filter.items()):
                continue
            yield t
            n += 1
            if limit is not None and n >= limit:
                return

    def put(self, config: RunnableConfig, checkpoint: Checkpoint, metadata: CheckpointMetadata,
            new_versions: Any) -> RunnableConfig:
        c = config["configurable"]
        thread_id, ns = c["thread_id"], c.get("checkpoint_ns", "")
        ctype, cblob = self.serde.dumps_typed(checkpoint)
        mtype, mblob = self.serde.dumps_typed(get_checkpoint_metadata(config, metadata))
        self._ns_ref(thread_id, ns).collection("checkpoints").document(checkpoint["id"]).set({
            "checkpoint_id": checkpoint["id"],
            "parent_checkpoint_id": c.get("checkpoint_id"),
            "type": ctype, "checkpoint": cblob,
            "metadata_type": mtype, "metadata": mblob,
        })
        return {"configurable": {"thread_id": thread_id, "checkpoint_ns": ns, "checkpoint_id": checkpoint["id"]}}

    def put_writes(self, config: RunnableConfig, writes: Sequence[tuple], task_id: str, task_path: str = "") -> None:
        c = config["configurable"]
        thread_id, ns, cid = c["thread_id"], c.get("checkpoint_ns", ""), c["checkpoint_id"]
        col = self._ns_ref(thread_id, ns).collection("writes")
        batch = self.db.batch()
        for idx, (channel, value) in enumerate(writes):
            vtype, vblob = self.serde.dumps_typed(value)
            widx = WRITES_IDX_MAP.get(channel, idx)
            batch.set(col.document(f"{cid}__{task_id}__{widx}"), {
                "checkpoint_id": cid, "task_id": task_id, "task_path": task_path,
                "idx": widx, "channel": channel, "type": vtype, "value": vblob,
            })
        batch.commit()

    def delete_thread(self, thread_id: str) -> None:
        self.db.recursive_delete(self.db.collection(self.collection).document(self._k(thread_id)))

    # ---- async API (the sync client is run in a worker thread) ----
    async def aget_tuple(self, config):
        return await asyncio.to_thread(self.get_tuple, config)

    async def alist(self, config, *, filter=None, before=None, limit=None):
        items = await asyncio.to_thread(lambda: list(self.list(config, filter=filter, before=before, limit=limit)))
        for item in items:
            yield item

    async def aput(self, config, checkpoint, metadata, new_versions):
        return await asyncio.to_thread(self.put, config, checkpoint, metadata, new_versions)

    async def aput_writes(self, config, writes, task_id, task_path=""):
        await asyncio.to_thread(self.put_writes, config, writes, task_id, task_path)


def firestore_checkpointer_builder(**kwargs):
    # Runs inside the deployed runtime. FirestoreSaver is pickled by value with this function.
    return FirestoreSaver(project_id=kwargs["project_id"], collection=kwargs.get("collection", "langgraph_checkpoints"))

FS_KWARGS = dict(project_id=PROJECT_ID, collection=FIRESTORE_COLL)
print("FirestoreSaver defined")

In [ ]:
# Local smoke test (Colab -> Firestore, your credentials): two turns on one thread, then read the checkpoints back
agent_fs = make_agent(firestore_checkpointer_builder, FS_KWARGS)
cfg = {"configurable": {"thread_id": "fs-local-1"}}
agent_fs.query(input={"role": "user", "content": "kanban"}, config=cfg)
agent_fs.query(input={"role": "user", "content": "compare it with scrum in one line"}, config=cfg)

saver = FirestoreSaver(PROJECT_ID, collection=FIRESTORE_COLL)
items = list(saver.list({"configurable": {"thread_id": "fs-local-1"}}))
latest = saver.get_tuple({"configurable": {"thread_id": "fs-local-1"}})
print("checkpoints stored:", len(items))
print("latest checkpoint id:", latest.config["configurable"]["checkpoint_id"] if latest else None)
RESULTS["4.1 local run, Firestore checkpointer"] = ("WORKS" if len(items) > 0 else "NO CHECKPOINTS", f"{len(items)} checkpoints")

In [ ]:
# Deploy: Firestore checkpointer behind the egress gateway (+ingress), token binding OFF.
# Firestore is a Google API over gRPC/HTTPS, so there is no raw DB wire protocol; whether the gateway carries it is the experiment.
cleanup_engines()     # one egress gateway per project+region: remove engines from earlier sections first
remote_fs = deploy("4.1 Firestore checkpointer + egress gateway + ingress, token binding OFF", agent_fs,
                   deploy_config("lab-4-firestore", egress_gw=EGRESS_GW_VPC, ingress_gw=INGRESS_GW, env=TOKEN_BINDING_OFF))
if remote_fs:
    ask(remote_fs, thread_id="fs-remote-1", label="4.1 Firestore checkpointer + egress gateway + ingress, token binding OFF")

### 4.2 Agent Runtime Sessions (not a LangGraph checkpointer)

Agent Runtime **Sessions** and **Memory Bank** are Google's managed state layer for the **ADK session model** (users, sessions, events). LangGraph's `checkpointer_builder` expects a `BaseCheckpointSaver`, and no Sessions-backed implementation exists in Google's docs. If you use Sessions, write conversation history at the application level. There is no code cell here.


### 4.3 Other options (not run here)

- **A thin service in front of the database.** A small Cloud Run (or MCP) service exposes `save_checkpoint` / `load_checkpoint` over HTTPS and talks to Cloud SQL itself. The agent calls it through the gateway like any other tool. The DB connection happens outside the governed path. This takes more code (a `BaseCheckpointSaver` that calls HTTP), so it is described rather than implemented.
- **AlloyDB** is Google's documented `checkpointer_builder` example. It is PostgreSQL-compatible, so it has the same wire-protocol problem behind an egress gateway; the private-IP approach in section 3 does not change that.


---
## Section 5: Cleanup

Deletes the engines this notebook tracked and, if you set the flags, the infrastructure it created. **Destructive:** the flags are `False` by default. Review each name before enabling.

In [ ]:
DELETE_ENGINES        = False   # engines created by this notebook (tracked in ENGINES)
DELETE_DATABASES      = False   # the two Cloud SQL instances and the secret
DELETE_NETWORK        = False   # attachment, template, VPC egress gateway, VPC, subnet, PSA range
DELETE_GATEWAYS       = False   # plain egress/ingress gateways, IAP extension and policies
DELETE_FIRESTORE_DATA = False   # the checkpoint collection in Firestore (not the database itself)

if DELETE_ENGINES:
    cleanup_engines()
if DELETE_DATABASES:
    for inst in (SQL_PUBLIC, SQL_PRIVATE):
        _sh(f"gcloud sql instances delete {inst} --quiet", check=False)
    _sh(f"gcloud secrets delete {SECRET_NAME} --quiet", check=False)
if DELETE_GATEWAYS:
    _sh(f"gcloud network-security authz-policies delete {EGRESS_GW}-authz-policy --location={REGION} --quiet", check=False)
    _sh(f"gcloud network-security authz-policies delete {EGRESS_GW_VPC}-authz-policy --location={REGION} --quiet", check=False)
    _sh(f"gcloud beta service-extensions authz-extensions delete {IAP_EXT} --location={REGION} --quiet", check=False)
    _sh(f"gcloud network-services agent-gateways delete {EGRESS_GW} --location={REGION} --quiet", check=False)
    _sh(f"gcloud network-services agent-gateways delete {INGRESS_GW} --location={REGION} --quiet", check=False)
if DELETE_NETWORK:
    _sh(f"gcloud network-services agent-gateways delete {EGRESS_GW_VPC} --location={REGION} --quiet", check=False)
    _sh(f"gcloud network-services agent-connectivity-templates delete {TEMPLATE_NAME} --location={REGION} --quiet", check=False)
    _sh(f"gcloud compute network-attachments delete {ATTACHMENT_NAME} --region={REGION} --quiet", check=False)
    _sh(f"gcloud services vpc-peerings delete --service=servicenetworking.googleapis.com --network={VPC_NAME} --quiet", check=False)
    _sh(f"gcloud compute addresses delete {PSA_RANGE_NAME} --global --quiet", check=False)
    _sh(f"gcloud compute networks subnets delete {SUBNET_NAME} --region={REGION} --quiet", check=False)
    _sh(f"gcloud compute networks delete {VPC_NAME} --quiet", check=False)
if DELETE_FIRESTORE_DATA:
    from google.cloud import firestore
    firestore.Client(project=PROJECT_ID).recursive_delete(firestore.Client(project=PROJECT_ID).collection(FIRESTORE_COLL))
print("cleanup flags processed (all False means nothing was deleted)")

---
## Section 6: Results and conclusion

In [ ]:
print(f"{'EXPERIMENT':<78} {'STATUS':<24} NOTES")
print("-" * 140)
for label in sorted(RESULTS):
    status, note = RESULTS[label]
    print(f"{label:<78} {status:<24} {note[:80]}")

### How to read your table

Compare your run with what we found. All rows below were reproduced by running the lab and by separate test deployments.

| Experiment | Outcome | Cause |
|------------|---------|-------|
| 1. No gateway, Cloud SQL checkpointer | Works | Default service identity, plain bearer token, direct connection |
| 2A. Egress + ingress, Agent Identity | `SSLV3_ALERT_HANDSHAKE_FAILURE` at `asyncpg.connect` | Gateway TLS inspection breaks the connector's mutual-TLS session to the instance |
| 2B. same + token-binding opt-out | Same failure as 2A | The opt-out does not matter once an egress gateway is in the path |
| 2C. Ingress only + `GOOGLE_API_PREVENT_AGENT_TOKEN_SHARING_FOR_GCP_SERVICES=false` | Works | Ingress does not touch outbound traffic |
| 2C'. Ingress only + `GOOGLE_API_ENABLE_RUNTIME_BOUND_TOKEN=false` | `401` from `sqladmin` | Token binding still applied; this variable is not the effective opt-out |
| 3. Private IP + VPC egress (PSC attachment) | Same TLS failure | Connector uses the private IP, but traffic is still intercepted: unused ports on the private IP accept TCP from a link-local address (inferred) |
| 4.1 Firestore checkpointer + egress gateway | Works | Google API over HTTPS/gRPC, no raw DB wire connection |
| 4.2 Sessions | Not a LangGraph checkpointer | ADK session model, no `BaseCheckpointSaver` documented |

### Conclusions

1. **Two separate problems hide behind "egress gateway breaks the SQL checkpointer".** (a) Without an egress gateway, Agent Identity tokens are bound to the runtime, so the Cloud SQL connector gets `401` from `sqladmin`; `GOOGLE_API_PREVENT_AGENT_TOKEN_SHARING_FOR_GCP_SERVICES=false` avoids that. (b) With an egress gateway, the connector's direct mutual-TLS database connection fails with a TLS handshake error.
2. **Neither problem is LangGraph's or Postgres's fault.** The same agent works with no gateway (section 1) and with ingress-only governance (2C).
3. **Egress governance and a direct database wire connection do not mix.** Private IP plus VPC egress (section 3) did not change that in our tests. AlloyDB speaks the same PostgreSQL protocol and has the same problem.
4. **The token-binding opt-out is a trade-off.** It removes protection against stolen token replay. Use it to learn; for production prefer paths that support bound tokens.
5. **Pick by what you need:**

| You need | Use |
|----------|-----|
| Simplest, no egress governance | Cloud SQL / AlloyDB checkpointer, no gateway (section 1) |
| Inbound governance only | Ingress gateway + `GOOGLE_API_PREVENT_AGENT_TOKEN_SHARING_FOR_GCP_SERVICES=false` (2C) |
| Full egress governance with checkpoints | HTTP-based store: Firestore saver (4.1), or a thin HTTPS service in front of the DB (4.3) |
| Managed ADK-style conversation memory | Agent Runtime Sessions / Memory Bank at the application level; not a LangGraph checkpointer |

### Caveats

- Test deployments are small and were run in one project and region; results reflect the platform at that time. Agent Gateway and Agent Runtime are evolving quickly, so re-run sections 2 and 3 to check newer behaviour.
- Section 3's interception explanation is inferred from network probes; the certificate issuer of the intercepting proxy was not obtained.
- The Firestore saver is subject to the 1 MiB document limit.
- Every deployment is cumulative in cost and time; run section 5 when you are done.
